# Intelligent Image Enhancement System for Low-Light Conditions & 3D Reconstruction
### מכללת סמי שמעון (SCE) - המחלקה להנדסת חשמל ואלקטרוניקה
**פרויקט מחקרי:** שחזור תלת-ממדי בתנאי שטח קיצוניים (חושך, רעש ומריחת תנועה)
**מגיש:** יחזקאל אבייב | **בהנחיית:** ד"ר עמית אפרים

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yehezkelab-lab/3D-image-reconstruction/blob/main/notebooks/3D_Reconstruction_Pipeline.ipynb)

---
### Pipeline Architecture:
1. **Step 1:** Importing pre-trained weights (`DarkIR`, `InstructIR`).
2. **Step 2:** Dual-Stream Deep Learning Enhancement (`DarkIR` + `InstructIR` with Tiling).
3. **Step 3:** Contrast-Limited Adaptive Histogram Equalization (`CLAHE`) in LAB color space.
4. **Step 4:** Sparse 3D Reconstruction using `SuperPoint` + `LightGlue` + `pycolmap` (`hloc`).
5. **Step 5:** Dense 3D Point Cloud Reconstruction using `COLMAP` undistorter and `CMVS-PMVS2`.


## Step 1: Importing Weights and Setting Up Dependencies
Mount Google Drive, install PyTorch/CV dependencies, and clone the neural enhancement repositories.

In [ ]:
# Step 1: Setup & Pretrained Weights
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')
%cd /content

!pip install -q pyiqa pytorch-msssim ptflops timm einops pycolmap kornia
!rm -rf /content/DarkIR /content/InstructIR
!git clone https://github.com/cidautai/DarkIR.git /content/DarkIR
!git clone https://github.com/mv-lab/InstructIR.git /content/InstructIR

os.makedirs('/content/DarkIR/models', exist_ok=True)
os.makedirs('/content/InstructIR/models', exist_ok=True)

weights = {
    'DarkIR_allLOL.pt': '/content/DarkIR/models/DarkIR_allLOL.pt',
    'im_instructir-7d.pt': '/content/InstructIR/models/im_instructir-7d.pt',
    'lm_instructir-7d.pt': '/content/InstructIR/models/lm_instructir-7d.pt'
}

for name, dest in weights.items():
    src = f'/content/drive/MyDrive/{name}'
    if os.path.exists(src):
        shutil.copyfile(src, dest)
        print(f'Loaded: {name}')
    else:
        print(f'Notice: {name} not found at {src}. Please ensure weights are in Drive or downloaded.')


## Step 2: Dual Neural Preprocessing Pipeline
Runs DarkIR for low-light recovery and InstructIR for motion deblurring.
Uses overlapping tiles (400x400 with 32px overlap) to prevent GPU memory exhaustion on high-resolution images.

In [ ]:
# Step 2: Dual Preprocessing pipeline (DarkIR + InstructIR with Tiling)
import sys
import os
import cv2
import numpy as np
import glob
import torch
import shutil
import yaml
import inspect
from torchvision import transforms
from PIL import Image
from google.colab import files

raw_dir = '/content/drive/MyDrive/Raw_Images'
output_dir = '/content/colmap_ready_dual'

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# --- Load DarkIR ---
os.chdir('/content/DarkIR')
if '/content/DarkIR' not in sys.path: sys.path.append('/content/DarkIR')
from archs.DarkIR import DarkIR

with open('options/test/NPE.yml', 'r') as f: opt = yaml.safe_load(f)
network_opt = opt.get('network_G', opt.get('network_g', opt.get('network', {})))
valid_params = inspect.signature(DarkIR.__init__).parameters
kwargs = {}
for param_name in valid_params:
    if param_name in ['self', 'args', 'kwargs']: continue
    val = network_opt.get(param_name, None)
    if param_name == 'in_channels' and 'inp_channels' in network_opt: val = network_opt['inp_channels']
    if param_name == 'inp_channels' and 'in_channels' in network_opt: val = network_opt['in_channels']
    if val is not None: kwargs[param_name] = val

model_dark = DarkIR(**kwargs)
checkpoint = torch.load('./models/DarkIR_allLOL.pt', map_location='cpu')
model_dark.load_state_dict(checkpoint.get('params_ema', checkpoint.get('params', checkpoint)), strict=True)
model_dark = model_dark.cuda().eval()

# --- Load InstructIR ---
os.chdir('/content/InstructIR')
if '/content/InstructIR' not in sys.path: sys.path.append('/content/InstructIR')

class dict2namespace:
    def __init__(self, config):
        for k, v in config.items():
            if isinstance(v, dict): self.__dict__[k] = dict2namespace(v)
            else: self.__dict__[k] = v

with open('configs/eval5d.yml', 'r') as f: config = yaml.safe_load(f)
cfg = dict2namespace(config)
from models import instructir
from text.models import LanguageModel, LMHead

model_instruct = instructir.create_model(
    input_channels=cfg.model.in_ch, width=cfg.model.width,
    enc_blks=cfg.model.enc_blks, middle_blk_num=cfg.model.middle_blk_num,
    dec_blks=cfg.model.dec_blks, txtdim=cfg.model.textdim
)
model_instruct.load_state_dict(torch.load('/content/InstructIR/models/im_instructir-7d.pt', map_location='cpu'), strict=True)
model_instruct.cuda().eval()

language_model = LanguageModel(model=cfg.llm.model).cuda().eval()
lm_head = LMHead(embedding_dim=cfg.llm.model_dim, hidden_dim=cfg.llm.embd_dim, num_classes=cfg.llm.nclasses)
lm_head.load_state_dict(torch.load('/content/InstructIR/models/lm_instructir-7d.pt', map_location='cpu'), strict=True)
lm_head.cuda().eval()

os.chdir('/content')

def run_darkir_func(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inp = transforms.ToTensor()(Image.fromarray(img_rgb)).unsqueeze(0).cuda()
    b, c, h, w = inp.shape
    tile, overlap = 400, 32
    stride = tile - overlap
    result, count = torch.zeros_like(inp), torch.zeros_like(inp)
    with torch.no_grad():
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                y2, x2 = min(y + tile, h), min(x + tile, w)
                y1, x1 = max(y2 - tile, 0), max(x2 - tile, 0)
                out = model_dark(inp[:, :, y1:y2, x1:x2], side_loss=False)
                if isinstance(out, (list, tuple)): out = out[0]
                result[:, :, y1:y2, x1:x2] += out
                count[:, :, y1:y2, x1:x2] += 1
    final_tensor = (result / count).clamp(0, 1).squeeze(0).cpu()
    return cv2.cvtColor(np.array(transforms.ToPILImage()(final_tensor)), cv2.COLOR_RGB2BGR)

def run_instructir_func(img_bgr):
    prompt = ["Please remove the blur and restore the image to be sharp."]
    with torch.no_grad():
        text_embd, _ = lm_head(language_model(prompt))
        text_embd = text_embd.cuda()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inp = transforms.ToTensor()(Image.fromarray(img_rgb)).unsqueeze(0).cuda()
    b, c, h, w = inp.shape
    tile, overlap = 400, 32
    stride = tile - overlap
    result, count = torch.zeros_like(inp), torch.zeros_like(inp)
    with torch.no_grad():
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                y2, x2 = min(y + tile, h), min(x + tile, w)
                y1, x1 = max(y2 - tile, 0), max(x2 - tile, 0)
                result[:, :, y1:y2, x1:x2] += model_instruct(inp[:, :, y1:y2, x1:x2], text_embd)
                count[:, :, y1:y2, x1:x2] += 1
    out_tensor = (result / count).clamp(0, 1).squeeze(0).cpu()
    return cv2.cvtColor(np.array(transforms.ToPILImage()(out_tensor)), cv2.COLOR_RGB2BGR)

image_files = [f for f in glob.glob(os.path.join(raw_dir, '**/*.*'), recursive=True) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Processing {len(image_files)} images...")
for idx, img_path in enumerate(image_files):
    name, ext = os.path.splitext(os.path.basename(img_path))
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: continue
    cv2.imwrite(os.path.join(output_dir, f"{name}_dark{ext}"), run_darkir_func(img_bgr))
    cv2.imwrite(os.path.join(output_dir, f"{name}_instruct{ext}"), run_instructir_func(img_bgr))

os.system(f'zip -q -r /content/images.zip {output_dir}')
print("Step 2 complete! Output saved to /content/images.zip")


## Step 3: Local Contrast Optimization (CLAHE)
Applies Contrast-Limited Adaptive Histogram Equalization on the Lightness channel (L) of the LAB color space.

In [ ]:
# Step 3: Histogram Equalization
import cv2
import os
import zipfile
import shutil

zip_path = '/content/images.zip'
extract_folder = '/content/extracted_images'
output_folder = '/content/fixed_images'

shutil.rmtree(extract_folder, ignore_errors=True)
shutil.rmtree(output_folder, ignore_errors=True)
os.makedirs(extract_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    exts = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    
    for root, _, files_in_dir in os.walk(extract_folder):
        for file in files_in_dir:
            if file.endswith(exts):
                img = cv2.imread(os.path.join(root, file))
                if img is None: continue
                
                lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
                l, a, b = cv2.split(lab)
                cl = clahe.apply(l)
                final = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
                cv2.imwrite(os.path.join(output_folder, file), final)
                
    shutil.make_archive('/content/fixed_images_archive', 'zip', output_folder)
    print("Step 3 complete! Saved /content/fixed_images_archive.zip")
else:
    print("Error: /content/images.zip not found.")


## Step 4: Sparse 3D Reconstruction
Extracts keypoints using **SuperPoint** and matches features with **LightGlue** via `hloc` and `pycolmap`.

In [ ]:
# Step 4: Sparse Reconstruction Pipeline
import os
import sys
import shutil
import re
from pathlib import Path

DRIVE_ZIP_PATH = '/content/fixed_images_archive.zip'
os.system("pip install -q pycolmap einops kornia")
os.system("pip install -q git+https://github.com/cvg/LightGlue.git")
os.system("pip install -q git+https://github.com/cvg/Hierarchical-Localization.git")
if not os.path.exists('/content/SuperGluePretrainedNetwork'):
    os.system("git clone https://github.com/magicleap/SuperGluePretrainedNetwork.git /content/SuperGluePretrainedNetwork > /dev/null 2>&1")

sys.path.insert(0, '/content')
from hloc import extract_features, match_features, reconstruction, pairs_from_exhaustive

IMAGE_DIR = Path('/content/input_images')
OUTPUT_DIR = Path('/content/hloc_outputs')
SFM_DIR = OUTPUT_DIR / 'sfm'

for p in [IMAGE_DIR, OUTPUT_DIR]:
    if p.exists(): shutil.rmtree(p)
IMAGE_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)

os.system(f"unzip -q -j {DRIVE_ZIP_PATH} -d {IMAGE_DIR}")
for filename in os.listdir(IMAGE_DIR):
    clean = re.sub(r'[^a-zA-Z0-9_.-]', '_', filename)
    if clean != filename:
        os.rename(IMAGE_DIR / filename, IMAGE_DIR / clean)

feature_conf = {
    'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
    'output': 'feats-superpoint-n4096-rmax1600',
    'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}
}

features_path = extract_features.main(feature_conf, IMAGE_DIR, feature_path=OUTPUT_DIR / 'features.h5')
pairs_path = OUTPUT_DIR / 'pairs-exhaustive.txt'
pairs_from_exhaustive.main(pairs_path, features=features_path)
matcher_conf = match_features.confs['superpoint+lightglue']
matches_path = match_features.main(matcher_conf, pairs_path, features=features_path, matches=OUTPUT_DIR / 'matches.h5')
SFM_DIR.mkdir(exist_ok=True)
reconstruction.main(SFM_DIR, IMAGE_DIR, pairs_path, features=features_path, matches=matches_path)

ZIP_OUTPUT = '/content/COLMAP_SPARSE_MODEL'
shutil.make_archive(ZIP_OUTPUT, 'zip', SFM_DIR)
print(f"Step 4 complete! Sparse model archived to {ZIP_OUTPUT}.zip")


## Step 5: Dense 3D Reconstruction
Corrects lens distortion with COLMAP `image_undistorter` and computes a dense point cloud using **CMVS-PMVS2**.

In [ ]:
# Step 5: Dense Reconstruction Pipeline
import os
import shutil
import zipfile
import glob
import subprocess
import re

os.chdir('/content')
IMG_DIR = '/content/input_images'
SPARSE_DIR = '/content/hloc_outputs/sfm'
PMVS_WORKSPACE = '/content/pmvs_workspace'
PMVS_TARGET = os.path.join(PMVS_WORKSPACE, 'pmvs')

os.system("sudo apt-get update -qq && sudo apt-get install -y colmap cmake build-essential libjpeg-dev libpng-dev libtiff-dev libgsl-dev liblapack-dev libblas-dev libx11-dev libglu1-mesa-dev freeglut3-dev -qq")
if not os.path.exists('/content/CMVS-PMVS'):
    os.system("git clone https://github.com/pmoulon/CMVS-PMVS.git > /dev/null 2>&1")
    os.makedirs('/content/CMVS-PMVS/program/build', exist_ok=True)
    os.chdir('/content/CMVS-PMVS/program/build')
    os.system("cmake .. > /dev/null 2>&1 && make -j4 > /dev/null 2>&1")

PMVS_BIN = '/content/CMVS-PMVS/program/build/main/pmvs2'
os.chdir('/content')

if os.path.exists(os.path.join(SPARSE_DIR, 'images.bin')):
    os.system(f"colmap model_converter --input_path {SPARSE_DIR} --output_path {SPARSE_DIR} --output_type TXT > /dev/null 2>&1")

imgs_txt = os.path.join(SPARSE_DIR, 'images.txt')
if os.path.exists(imgs_txt):
    phys_files = {re.sub(r'[^a-zA-Z0-9]', '', f).lower(): os.path.join(r, f) for r, _, fs in os.walk(IMG_DIR) for f in fs}
    with open(imgs_txt, 'r') as f: lines = f.readlines()
    new_lines, is_img = [], True
    for line in lines:
        if line.startswith('#'):
            new_lines.append(line)
            continue
        if is_img:
            parts = line.strip().split()
            if len(parts) >= 10:
                orig = " ".join(parts[9:])
                safe = re.sub(r'[^a-zA-Z0-9.]', '_', orig).lower()
                sim = re.sub(r'[^a-zA-Z0-9]', '', orig).lower()
                if sim in phys_files:
                    shutil.move(phys_files[sim], os.path.join(IMG_DIR, safe))
                line = " ".join(parts[:9] + [safe]) + "\n"
            new_lines.append(line)
        is_img = not is_img
    with open(imgs_txt, 'w') as f: f.writelines(new_lines)

subprocess.run(["colmap", "image_undistorter", "--image_path", IMG_DIR, "--input_path", SPARSE_DIR, "--output_path", PMVS_WORKSPACE, "--output_type", "PMVS"], check=True)
opt_file = os.path.join(PMVS_TARGET, 'option-all')
txt_dir = os.path.join(PMVS_TARGET, 'txt')
valid = [int(f.split('.')[0]) for f in os.listdir(txt_dir) if f.endswith('.txt') and os.path.getsize(os.path.join(txt_dir, f)) > 0]
if os.path.exists(opt_file):
    with open(opt_file, 'r') as f: lines = f.read().splitlines()
    with open(opt_file, 'w') as f:
        f.write("\n".join([f"timages {len(valid)} " + " ".join(map(str, valid)) if l.strip().startswith('timages') else l for l in lines]) + "\n")

if subprocess.run([PMVS_BIN, f"{PMVS_TARGET}/", "option-all"]).returncode == 0:
    ply = os.path.join(PMVS_TARGET, 'models', 'option-all.ply')
    if os.path.exists(ply):
        shutil.make_archive('/content/FINAL_DENSE', 'zip', os.path.dirname(ply), os.path.basename(ply))
        print(f"Success! Dense point cloud generated: {ply}")
        print("Download /content/FINAL_DENSE.zip to view 3D reconstruction.")
